In [36]:
## IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta

snp_raw = pd.read_csv("data/compustat_fundamental_annual.csv")

In [37]:
snp_raw["sic"].value_counts()

sic
6722    36326
6726     6924
6020     6428
2836     6360
7370     4437
        ...  
800         4
5094        4
9998        3
5141        3
5020        2
Name: count, Length: 411, dtype: int64

- gvkey: company key, conm: company name

In [38]:
# drop costat column since it is not relevant to our analysis
snp_raw.drop("costat", inplace=True, axis=1)
# drop CAD since we dont want to convert it to USD
snp_raw.drop(snp_raw[snp_raw["curcd"] == "CAD"].index, inplace=True, axis=0)
# drop curcd column since we only have USD data
snp_raw.drop("curcd", inplace=True, axis=1)
# drop datafmt, indfmt and consol columns since they are not relevant to our analysis
snp_raw.drop(["datafmt", "indfmt", "consol"], inplace=True, axis=1)
# drop fyr column since we can calculate fiscal year end date using datadate column
snp_raw.drop("fyr", inplace=True, axis=1)

In [39]:
snp_raw.head()

,gvkey,datadate,conm,tic,cik,naics,sic,at,ceq,che,...,ebit,ebitda,gp,ni,revt,sale,capx,oancf,csho,prcc_f
0,1004,31/5/2009,AAR CORP,AIR,1750.0,423860.0,5080,1377.511,656.895,112.505,...,125.529,166.080,313.299,78.651,1423.976,1423.976,27.535,64.451,38.884,14.70
1,1004,31/5/2010,AAR CORP,AIR,1750.0,423860.0,5080,1501.042,746.906,79.370,...,95.415,134.345,286.249,44.628,1352.151,1352.151,28.855,153.156,39.484,19.70
2,1004,31/5/2011,AAR CORP,AIR,1750.0,423860.0,5080,1703.727,835.845,57.433,...,137.016,196.312,367.711,69.826,1775.782,1775.782,124.879,108.598,39.781,26.39
3,1004,31/5/2012,AAR CORP,AIR,1750.0,423860.0,5080,2195.653,864.649,67.720,...,142.360,222.693,412.090,67.723,2074.498,2074.498,91.218,94.217,40.273,12.05
4,1004,31/5/2013,AAR CORP,AIR,1750.0,423860.0,5080,2136.900,918.600,75.300,...,136.600,245.200,452.600,55.000,2167.100,2167.100,37.600,162.900,39.382,20.06


In [2]:
import pandas as pd

# --------------------------------------------------
# 1. Read S&P 500 constituents file
# --------------------------------------------------
sp500 = pd.read_csv("data/constituents.csv")

# Check the first few rows
sp500.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [4]:
# --------------------------------------------------
# 2. Remove companies under Financials sector
# --------------------------------------------------
sp500_nonfin = sp500[
    sp500["GICS Sector"] != "Financials"
].copy()

print("Original number of S&P 500 companies:", len(sp500))
print("After removing Financials:", len(sp500_nonfin))

Original number of S&P 500 companies: 503
After removing Financials: 427


In [5]:
# --------------------------------------------------
# 3. Remove banking-related companies using sub-industry keywords
# --------------------------------------------------
banking_keywords = [
    "bank",
    "banks",
    "banking",
    "financial",
    "insurance",
    "capital markets",
    "mortgage",
    "consumer finance",
    "diversified financial",
    "asset management",
    "investment banking",
    "brokerage"
]

pattern = "|".join(banking_keywords)

sp500_nonfin = sp500_nonfin[
    ~sp500_nonfin["GICS Sub-Industry"]
    .str.lower()
    .str.contains(pattern, na=False)
].copy()

print("After removing Financials and banking-related firms:", len(sp500_nonfin))

After removing Financials and banking-related firms: 427


In [6]:
# --------------------------------------------------
# 4. Create Yahoo Finance-compatible ticker
# --------------------------------------------------
sp500_nonfin["ticker_yahoo"] = (
    sp500_nonfin["Symbol"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(".", "-", regex=False)
)

sp500_nonfin[["Symbol", "ticker_yahoo", "Security", "GICS Sector", "GICS Sub-Industry"]].head()

,Symbol,ticker_yahoo,Security,GICS Sector,GICS Sub-Industry
0,MMM,MMM,3M,Industrials,Industrial Conglomerates
1,AOS,AOS,A. O. Smith,Industrials,Building Products
2,ABT,ABT,Abbott Laboratories,Health Care,Health Care Equipment
3,ABBV,ABBV,AbbVie,Health Care,Biotechnology
4,ACN,ACN,Accenture,Information Technology,IT Consulting & Other Services
